# 02 · Four transforms and paired chains

Use one embedding and change the transform. Coefficient axes retain their own
meanings: cycles per residue, log-frequency, or wavelet band/position. Requires
`Protspect[ml,wavelets,validation]` from the development checkout. No external
data are needed. These illustrative sequences carry no biological labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from protspect import transform_sequence, transform_pair, ComplexSet
from protspect.embeddings import kidera_factors
%matplotlib inline

In [ ]:
SEQUENCE_A = "IKLFYRACDEFGHIK"
SEQUENCE_B = "ACDEFGHIKLFYR"
EMBEDDING = kidera_factors()
LENGTH = 64
CHANNEL = "KF1"
CENTER = False
NORMALIZATION = "energy"
OPTIONS = {"fourier": {}, "z": {"radius": 1.05},
    "mellin": {"mellin_c": 0.0},
    "wavelet": {"wavelet": "haar", "level": 3, "mode": "periodization"}}
COMPONENT = "real"  # amplitude, phase, real, imaginary

In [ ]:
results = {method: transform_sequence(SEQUENCE_A, EMBEDDING, method=method,
    length=LENGTH, center=CENTER, normalization=NORMALIZATION, **options)
    for method, options in OPTIONS.items()}
component = {"amplitude": np.abs, "phase": np.angle,
    "real": np.real, "imaginary": np.imag}[COMPONENT]
fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
for ax, (method, result) in zip(axes.flat, results.items()):
    if method == "wavelet":
        x, label = np.arange(len(result.coefficients[CHANNEL])), "Concatenated band coefficient"
    elif method == "mellin":
        x, label = result.coordinates["log_frequency"], "Log-frequency ν"
    else:
        x, label = result.coordinates["frequency"], "Cycles per residue"
    ax.plot(x, component(result.coefficients[CHANNEL]))
    ax.set(title=method, xlabel=label, ylabel=COMPONENT)
plt.show()
unit_z = transform_sequence(SEQUENCE_A, EMBEDDING, method="z", radius=1.,
    length=LENGTH, center=CENTER, normalization=NORMALIZATION)
np.testing.assert_allclose(unit_z.coefficients[CHANNEL], results["fourier"].coefficients[CHANNEL])

## Inspect wavelet bands and channel products

Magnitudes discard wavelet signs. Signed real coefficients preserve them. Channel
products are computed at corresponding coordinates, without statistical averaging.

In [ ]:
wavelet = results["wavelet"]
mask = wavelet.coordinates["band"] == "D1"
values, names = wavelet.features("real", selector=lambda c: c["band"] == "D1")
assert len(values) == len(EMBEDDING) * mask.sum()
print("Finest-band features:", len(values))
cross = wavelet.cross()
np.testing.assert_allclose(cross.coefficients["KF1_KF2"],
    wavelet.coefficients["KF1"] * wavelet.coefficients["KF2"].conj())

## Keep ComplexSet for paired workflows

Choose any method below. A/B chains share one grid; their preprocessing is
independent. Every A-channel is multiplied by every conjugated B-channel.
These descriptors do not infer three-dimensional contacts or binding energies.

In [ ]:
METHOD = "wavelet"
pairs = ComplexSet(embedding=EMBEDDING, method=METHOD,
    transform_options=OPTIONS[METHOD], center=CENTER, normalization=NORMALIZATION)
pairs.from_sequence_list([SEQUENCE_A], [SEQUENCE_B], length=LENGTH,
    attributes={"pair_id": ["example-pair"]})
paired_table = pairs.export_features(COMPONENT, attributes=["pair_id"])
print("Paired table:", paired_table.shape)
display(paired_table.iloc[:, :8])
print("Chain A:", pairs.export_features(COMPONENT, chain_ID="A").shape)
forward = transform_pair(SEQUENCE_A, SEQUENCE_B, EMBEDDING, method=METHOD,
    length=LENGTH, center=CENTER, normalization=NORMALIZATION, **OPTIONS[METHOD])
backward = transform_pair(SEQUENCE_B, SEQUENCE_A, EMBEDDING, method=METHOD,
    length=LENGTH, center=CENTER, normalization=NORMALIZATION, **OPTIONS[METHOD])
np.testing.assert_allclose(forward.coefficients["A-KF1_B-KF2"],
                          backward.coefficients["A-KF2_B-KF1"].conj())

## Interpret the changes

Z radius 1 equals Fourier; other radii change positional weighting. Mellin here
is a finite weighted sum on positive positions, not automatic scale invariance.
Wavelet filter, level, and boundary mode affect features. Change one setting at
a time and inspect the stored coordinates/metadata before training a model.
Notebook 03 demonstrates fitting inside cross-validation.